# 13장 실습 — 시뮬 데이터를 실기처럼 고치기

12장은 시뮬 데이터와 실기 데이터를 **섞었다.** 시뮬이 상태 공간을 덮고 실기가 어긋남을 맞췄다. 그리고 맞추는 데 실기 시연 스무 판이 들었다.

이 장은 다른 길을 간다. 섞는 대신 **시뮬 데이터 자체를 고친다.** 실기에서 얻은 소량의 정보로 시뮬 데이터를 실기처럼 보이게 바꾼 뒤, 그 고친 데이터만으로 학습시킨다.

실제 시스템에서 이 일을 하는 것이 생성 모델이다. RCAN 계열은 실기 영상을 시뮬의 「표준 모습」으로 되돌리고, Cosmos-Transfer 계열은 깊이·분할 같은 제어 입력을 붙들고 시뮬 영상을 실기처럼 다시 그린다. 이 노트북은 영상 대신 저차원 관측으로 **그 구조**를 재현한다 — 무엇이 번역을 작동하게 하고 무엇이 망가뜨리는지가 차원 수와 무관하기 때문이다.

실행 시간은 CPU에서 3분 안팎이다.

In [1]:
%pip install -q mujoco==3.13.0 koreanize-matplotlib

Note: you may need to restart the kernel to use updated packages.


In [2]:
import time
import unicodedata
import numpy as np
import mujoco
import torch
import torch.nn as nn
import matplotlib.pyplot as plt
import koreanize_matplotlib  # noqa: F401


def dw(s):
    return sum(2 if unicodedata.east_asian_width(c) in "WF" else 1
               for c in str(s))


def pad(s, n, right=False):
    gap = " " * max(n - dw(s), 0)
    return gap + str(s) if right else str(s) + gap


RED, GREY, INK = "#E32C24", "#8C8A85", "#1C1C1A"
PINK = "#F3A29D"
plt.rcParams.update({"figure.dpi": 110, "axes.grid": True,
                     "grid.alpha": .25, "axes.spines.top": False,
                     "axes.spines.right": False})
print("mujoco", mujoco.__version__, "/ torch", torch.__version__)

mujoco 3.13.0 / torch 2.14.0+cu130


## 두 도메인

시뮬과 실기를 세운다. 앞 장들과 같은 방식인데, 이번에는 차이에 **관측 어긋남**을 하나 더 넣는다.

- 시뮬 — 마찰 0.60, 게인 120, 지연 없음, 잡음 없음, 관측 정확
- 실기 — 마찰 0.75, 게인 100, 지연 1스텝, 관측 잡음 3 mm, 그리고 **블록 위치 관측이 (+15, −10) mm 어긋나 있다**

마지막 항이 이 노트북의 장치다. 실제 시스템에서 늘 생기는 교정 오차를 흉내 낸 것이다 — 카메라 외부 파라미터가 조금 틀렸거나, 마커가 물체 중심에서 벗어나 붙었거나 하면 이렇게 된다. 시뮬에는 이 어긋남이 없다.

전문가는 6장에서 쓴 서보 규칙이고, **참값**을 보고 행동한다. 사람이 모션캡처를 보며 원격조작하는 상황에 해당한다.

12장에서 이 설정으로 잰 결과가 출발점이다 — 시뮬 데이터만으로 학습한 정책의 실기 성공률이 **0.00**이었고, 실기 스무 판을 섞어야 0.9대가 나왔다.

In [3]:
XML = '''
<mujoco model="planar_push">
  <option timestep="0.004" integrator="implicitfast"/>
  <worldbody>
    <geom name="floor" type="plane" size="3 3 .1"
          friction="{mu} .005 .0001"/>
    <body name="block" pos="0 0 .031">
      <freejoint/>
      <geom type="cylinder" size=".03 .03" mass=".25"
            friction="{mu} .005 .0001"/>
    </body>
    <body name="pusher" pos="0 0 .03">
      <joint name="px" type="slide" axis="1 0 0"/>
      <joint name="py" type="slide" axis="0 1 0"/>
      <geom type="cylinder" size=".012 .03" mass="4"/>
    </body>
  </worldbody>
  <actuator>
    <velocity joint="px" kv="{kv}"/>
    <velocity joint="py" kv="{kv}"/>
  </actuator>
</mujoco>
'''

EP, GOAL_R, AMAX = 110, .03, .35
SIM = dict(mu=.60, kv=120., lat=0, noise=0., bias=(0., 0.))
REAL = dict(mu=.75, kv=100., lat=1, noise=.003, bias=(.015, -.010))
CACHE = {}


def model_of(mu, kv):
    k = (round(mu, 2), round(kv, 0))
    if k not in CACHE:
        CACHE[k] = mujoco.MjModel.from_xml_string(
            XML.format(mu=k[0], kv=k[1]))
    return CACHE[k]


def unit(v):
    n = np.linalg.norm(v)
    return v / n if n > 1e-9 else np.zeros_like(v)


def expert(b, p, g):
    # 6장의 서보 규칙. 참값을 보고 행동한다
    dv = unit(g - b)
    if np.linalg.norm(g - b) < .012:
        return np.zeros(2)
    e = (b - dv * .045) - p
    if np.linalg.norm(e) > .013:
        u = 14. * e
        n = np.linalg.norm(u)
        return u / n * .22 if n > .22 else u
    return dv * .22


def rollout(dom, rng, pol=None, keep=False):
    m = model_of(dom["mu"], dom["kv"])
    d = mujoco.MjData(m)
    sub = int(round(1 / 20 / m.opt.timestep))
    bx, by = rng.uniform(-.03, .03, 2)
    d.qpos[0:2] = [bx, by]
    d.qpos[2] = .031
    d.qpos[7:9] = [bx - .09, by]
    mujoco.mj_forward(m, d)
    g = np.array([.30, 0.]) + rng.uniform(-.04, .04, 2)
    buf = [np.zeros(2)] * (dom["lat"] + 1)
    log = []
    for t in range(EP):
        b_true = np.asarray(d.qpos[0:2], float)
        b = b_true + np.asarray(dom["bias"])     # 관측만 어긋난다
        if dom["noise"]:
            b = b + rng.normal(0, dom["noise"], 2)
        o = np.concatenate([g - b, d.qvel[0:2],
                            b - d.qpos[7:9],
                            d.qvel[6:8]]).astype(np.float32) * 10.
        if pol is None:
            a = expert(b_true, np.asarray(d.qpos[7:9], float), g)
        else:
            a = pol(o)
        a = np.clip(a, -AMAX, AMAX)
        if keep:
            # 같은 장면의 「표준(시뮬) 관측」을 함께 적어 둔다
            oc = np.concatenate(
                [g - b_true, d.qvel[0:2], b_true - d.qpos[7:9],
                 d.qvel[6:8]]).astype(np.float32) * 10.
            log.append((o, a / AMAX, oc))
        buf.append(a)
        d.ctrl[:] = buf[-1 - dom["lat"]]
        for _ in range(sub):
            mujoco.mj_step(m, d)
    ok = np.linalg.norm(d.qpos[0:2] - g) < GOAL_R
    return (ok, log) if keep else ok


def collect(dom, n, seed):
    # 성공한 시연만 모은다
    X, Y, C = [], [], []
    rng = np.random.default_rng(seed)
    got = tries = 0
    while got < n and tries < n * 4:
        tries += 1
        ok, log = rollout(dom, rng, keep=True)
        if not ok:
            continue
        got += 1
        for o, a, oc in log:
            X.append(o)
            Y.append(a)
            C.append(oc)
    return (np.array(X, np.float32), np.array(Y, np.float32),
            np.array(C, np.float32), got)

## 데모를 모으고 기준선을 다시 잡는다

`collect` 가 한 가지를 더 기록한다. 실기 시연의 매 스텝마다 **같은 순간의 「표준 관측」** — 어긋남과 잡음이 없는, 시뮬이라면 보였을 관측 — 을 함께 적어 둔다.

실기에서 이것을 얻을 수 있느냐가 이 장의 갈림길이다. 영상이라면 같은 장면의 깊이 지도나 분할 마스크가 그 역할을 한다. 어느 쪽이든 **실기 관측과 표준 표현을 짝지을 수 있느냐**가 핵심이고, 뒤에서 그것이 없을 때 무슨 일이 생기는지도 본다.

In [4]:
t0 = time.time()
Xs, Ys, Cs, ns_ep = collect(SIM, 400, 1)
Xr, Yr, Cr, nr_ep = collect(REAL, 20, 2)
print(f"시뮬 {ns_ep}판 {len(Xs):,}샘플"
      f" / 실기 {nr_ep}판 {len(Xr):,}샘플"
      f"  ({time.time()-t0:.0f}s)")


class BC(nn.Module):
    def __init__(self):
        super().__init__()
        self.f = nn.Sequential(nn.Linear(8, 64), nn.Tanh(),
                               nn.Linear(64, 64), nn.Tanh(),
                               nn.Linear(64, 2))

    def forward(self, x):
        return self.f(x)


def train_bc(X, Y, seed=0, epochs=60):
    torch.manual_seed(seed)
    net = BC()
    opt = torch.optim.Adam(net.parameters(), 1e-3)
    X, Y = torch.from_numpy(X), torch.from_numpy(Y)
    n = len(X)
    for _ in range(epochs):
        idx = torch.randperm(n)
        for k in range(0, n, 256):
            j = idx[k:k + 256]
            loss = ((net(X[j]) - Y[j]) ** 2).mean()
            opt.zero_grad()
            loss.backward()
            opt.step()
    return net


def evaluate(net, dom, n=120, seed=999):
    rng = np.random.default_rng(seed)

    def pol(o):
        with torch.no_grad():
            return net(torch.from_numpy(o)).numpy() * AMAX
    return float(np.mean([rollout(dom, rng, pol=pol)
                          for _ in range(n)]))


rng = np.random.default_rng(7)
IDX = rng.integers(0, len(Xs), 20_000)
base = evaluate(train_bc(Xs[IDX], Ys[IDX]), REAL)
print(f"시뮬 데이터 그대로 학습 → 실기 성공률 {base:.2f}")

시뮬 400판 44,000샘플 / 실기 20판 2,200샘플  (7s)


시뮬 데이터 그대로 학습 → 실기 성공률 0.00


## 짝이 있을 때 — 사상을 배운다

실기 시연에서 얻은 것은 같은 순간의 두 관측이다. 하나는 실기가 실제로 내놓은 것이고 다른 하나는 시뮬이라면 내놓았을 것이다. 이 짝에서 **표준 → 실기** 사상을 배운다.

여기서는 최소제곱으로 아핀 사상 하나를 맞춘다. 실제 시스템의 생성 모델이 하는 일도 형식은 같다 — 표준 표현을 조건으로 받아 목표 도메인의 모습을 내놓는다. 다만 그쪽은 사상이 픽셀 수준으로 복잡해서 큰 신경망이 필요할 뿐이다.

사상을 배우고 나면 **시뮬 데이터 44,000개 전부**를 그 사상에 통과시킨다. 실기 시연은 사상을 배우는 데만 쓰이고 학습 데이터에는 한 줄도 들어가지 않는다.

In [5]:
def fit_paired(C, X, k):
    # (표준 관측, 실기 관측) 짝 k 개로 아핀 사상을 맞춘다
    A = np.hstack([C[:k], np.ones((k, 1), np.float32)])
    W = np.linalg.lstsq(A, X[:k], rcond=None)[0]

    def tr(Z):
        Zb = np.hstack([Z, np.ones((len(Z), 1), np.float32)])
        return (Zb @ W).astype(np.float32)
    return tr


rows = []
for k_ep in (1, 5, 20):
    tr = fit_paired(Cr, Xr, k_ep * EP)
    net = train_bc(tr(Xs[IDX]), Ys[IDX])
    rows.append((k_ep, evaluate(net, REAL)))
    print(f"짝 {k_ep}판 끝  ({time.time()-t0:.0f}s)")

print()
print(pad("방법", 26) + pad("쓴 실기 시연", 14, True)
      + pad("실기 성공률", 14, True))
print(pad("시뮬 그대로", 26) + pad("0판", 14, True)
      + pad(f"{base:.2f}", 14, True))
for k_ep, v in rows:
    print(pad("짝으로 번역", 26) + pad(f"{k_ep}판", 14, True)
          + pad(f"{v:.2f}", 14, True))

짝 1판 끝  (22s)


짝 5판 끝  (28s)


짝 20판 끝  (35s)

방법                        쓴 실기 시연   실기 성공률
시뮬 그대로                          0판          0.00
짝으로 번역                          1판          0.83
짝으로 번역                          5판          0.78
짝으로 번역                         20판          0.77


## 짝이 없을 때 — 분포만 맞춰 본다

짝을 얻기 어려운 경우가 훨씬 흔하다. 실기 영상과 시뮬 영상이 있는데 같은 장면이 아닌 경우다. 그럴 때 쓰는 것이 분포 정합이고, 짝 없는 영상 번역(CycleGAN 계열)이 대표적이다.

여기서는 가장 단순한 형태로 해 본다. 시뮬 관측의 평균과 표준편차를 실기 관측의 것에 맞춘다.

In [6]:
def fit_unpaired(Xs_, Xr_):
    ms, ss = Xs_.mean(0), Xs_.std(0) + 1e-6
    mr, sr = Xr_.mean(0), Xr_.std(0) + 1e-6
    return lambda Z: ((Z - ms) / ss * sr + mr).astype(np.float32)


tr_u = fit_unpaired(Xs, Xr)
net_u = train_bc(tr_u(Xs[IDX]), Ys[IDX])
print(f"짝 없이 분포만 정합 → 실기 성공률 "
      f"{evaluate(net_u, REAL):.2f}")

zero = np.zeros((1, 8), np.float32)
shift_u = tr_u(zero)[0] / 10. * 1000
shift_p = fit_paired(Cr, Xr, 20 * EP)(zero)[0] / 10. * 1000
names = ["목표−블록 x", "목표−블록 y", "블록속도 x", "블록속도 y",
         "블록−밀대 x", "블록−밀대 y", "밀대속도 x", "밀대속도 y"]
print()
print(pad("관측 성분", 16) + pad("참 어긋남", 12, True)
      + pad("짝으로", 12, True) + pad("짝 없이", 12, True))
truth = [-15., 10., 0., 0., 15., -10., 0., 0.]
for nm, t_, p_, u_ in zip(names, truth, shift_p, shift_u):
    print(pad(nm, 16) + pad(f"{t_:+.0f}", 12, True)
          + pad(f"{p_:+.1f}", 12, True)
          + pad(f"{u_:+.1f}", 12, True))

짝 없이 분포만 정합 → 실기 성공률 0.15

관측 성분          참 어긋남      짝으로     짝 없이
목표−블록 x              -15       -15.1        +2.0
목표−블록 y              +10        +9.9        +9.2
블록속도 x                +0        +0.0        +2.0
블록속도 y                +0        -0.0        -1.2
블록−밀대 x              +15       +15.1       -15.3
블록−밀대 y              -10        -9.9        -9.7
밀대속도 x                +0        +0.0        -3.9
밀대속도 y                +0        +0.0        -1.6


## 정리

**번역은 혼합보다 실기 데이터를 훨씬 적게 먹는다.** 12장에서 스무 판을 섞어 얻은 성적을, 여기서는 짝 한 판으로 얻었다. 실기 데이터를 정책에 먹이지 말고 번역기에 먹인다는 규칙이 그 차이를 만든다.

**까닭은 배워야 할 사상의 크기다.** 혼합에서 실기 데이터는 「관측 → 행동」 전체를 고쳐야 하고, 번역에서는 「표준 관측 → 실기 관측」만 알려 주면 된다. 뒤쪽이 훨씬 작다.

**짝이 없으면 번역이 망가진다.** 분포 차이에는 고치고 싶은 도메인 차이와 고치면 안 되는 행동 차이가 섞여 있고, 분포만 맞추는 방법은 그 둘을 못 가른다. 추정된 어긋남의 부호가 뒤집히는 것이 그 증상이다.

**그래서 제어 입력을 건다.** 깊이·분할처럼 도메인이 바뀌어도 변하지 않는 표현을 조건으로 걸면 번역기가 모습만 바꾸고 내용은 못 바꾼다. 최근 시스템들이 공통으로 쓰는 장치다.

다음 장은 시뮬 데이터의 다른 축을 본다. 지금까지는 **어떻게 보이는가**를 다뤘다. 다음 질문은 **무엇을 시키는가**다.